In [1]:
# Block 0 — setup. Code from GitHub onto the VM disk; data + embeddings on Drive.
import os, sys
from google.colab import drive
drive.mount('/content/drive')

REPO_DIR = "/content/NeoAntigen"
try:   # private repo: store a GitHub token as Colab secret GH_TOKEN
    from google.colab import userdata
    TOKEN = userdata.get("GH_TOKEN")
except Exception:
    TOKEN = ""
URL = f"https://{TOKEN + '@' if TOKEN else ''}github.com/Tienanh209/NeoAntigen.git"
if not os.path.exists(REPO_DIR):
    !git clone {URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only
%cd {REPO_DIR}
sys.path.insert(0, REPO_DIR)

# numpy/scipy/torch are filtered out: Colab's torch is built against its own numpy
!grep -viE '^(torch|torchvision|torchaudio|nvidia-|numpy|scipy|#)' requirements.txt > req_colab.txt
!pip install -q -r req_colab.txt

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Already up to date.
/content/NeoAntigen


In [2]:
# Block 1 — dataset and its build id (the cache key)
import torch
from src import config, data, embed

df = data.load_dataset()
BID = data.build_id(df)
print("DATA_DIR :", config.DATA_DIR)
print("build id :", BID, "  (strict and relaxed builds get different ids)")
print("sequences:", len(df), "| residues:", int(df["length"].sum()))
print("GPU      :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

DATA_DIR : /content/drive/MyDrive/s1156052/NeoAntigen/data
build id : e2a2999ece   (strict and relaxed builds get different ids)
sequences: 1793 | residues: 51413
GPU      : NVIDIA A100-SXM4-40GB


In [3]:
# Block 2 — SMOKE TEST: tiny 8M checkpoint, a few seconds.
# Run this once to prove the cache layout, shapes and Drive writes before
# committing to the 650M models.
embed.build_cache("esm2_8m", df, build_id=BID)
c = embed.EmbCache("esm2_8m", BID, staged=False)
x, lens = c.stack(df["uid"].head(4), data.max_len(df))
print("stack:", x.shape, "lengths:", lens, "| dim:", c.dim)

[esm2_8m] loading facebook/esm2_t6_8M_UR50D on cuda


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/775 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 31.4MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[esm2_8m] shard 1/1  64/1793
[esm2_8m] shard 1/1  704/1793
[esm2_8m] shard 1/1  1344/1793
[esm2_8m] cached 1793 sequences, 51413 residues, 33 MB -> /content/drive/MyDrive/s1156052/NeoAntigen/data/emb/e2a2999ece/esm2_8m
stack: (4, 320, 35) lengths: [18 31 31 20] | dim: 320


In [4]:
# Block 3 — ProtTrans (ProtT5-XL, encoder-only fp16, 1024-d). The paper's primary feature set.
embed.build_cache("prott5", df, build_id=BID)

[prott5] loading Rostlab/prot_t5_xl_half_uniref50-enc on cuda


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/238k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/1.79k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/656 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.42GB            

model.safetensors: downloading bytes:           |  0.00B            

[prott5] shard 1/1  64/1793
[prott5] shard 1/1  704/1793
[prott5] shard 1/1  1344/1793
[prott5] cached 1793 sequences, 51413 residues, 105 MB -> /content/drive/MyDrive/s1156052/NeoAntigen/data/emb/e2a2999ece/prott5


PosixPath('/content/drive/MyDrive/s1156052/NeoAntigen/data/emb/e2a2999ece/prott5')

In [5]:
# Block 4 — ESM-2 650M (1280-d)
embed.build_cache("esm2_650m", df, build_id=BID)

[esm2_650m] loading facebook/esm2_t33_650M_UR50D on cuda


config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/534 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t33_650M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[esm2_650m] shard 1/1  64/1793
[esm2_650m] shard 1/1  704/1793
[esm2_650m] shard 1/1  1344/1793
[esm2_650m] cached 1793 sequences, 51413 residues, 132 MB -> /content/drive/MyDrive/s1156052/NeoAntigen/data/emb/e2a2999ece/esm2_650m


PosixPath('/content/drive/MyDrive/s1156052/NeoAntigen/data/emb/e2a2999ece/esm2_650m')

In [6]:
# Block 5 — ESM-1b (1280-d)
embed.build_cache("esm1b", df, build_id=BID)

[esm1b] loading facebook/esm1b_t33_650M_UR50S on cuda


config.json:   0%|          | 0.00/782 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

...

[transformers] EsmModel LOAD REPORT from: facebook/esm1b_t33_650M_UR50S
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.decoder.weight    | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors: reconstructing file:   0%|          |  0.00B / 2.61GB            

model.safetensors: downloading bytes:           |  0.00B            

[esm1b] shard 1/1  64/1793
[esm1b] shard 1/1  704/1793
[esm1b] shard 1/1  1344/1793
[esm1b] cached 1793 sequences, 51413 residues, 132 MB -> /content/drive/MyDrive/s1156052/NeoAntigen/data/emb/e2a2999ece/esm1b


PosixPath('/content/drive/MyDrive/s1156052/NeoAntigen/data/emb/e2a2999ece/esm1b')

In [7]:
# Block 6 — what is cached
for plm in ["prott5", "esm2_650m", "esm1b", "esm2_8m"]:
    ok = embed.is_cached(plm, BID)
    print(f"{plm:12s} {'cached' if ok else 'MISSING'}")

prott5       cached
esm2_650m    cached
esm1b        cached
esm2_8m      cached
